In [1]:
import pandas as pd
import numpy as np
import glob
import warnings

In [2]:
# Suppress the openpyxl unknown extension warnings to keep your output clean
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [3]:
# ==========================================
# 1. The Ultimate Master Column Map
# ==========================================
column_map = {
    'fname': 'First Name', 'lname': 'Last Name', 'Name': 'First Name', 
    'bib': 'Bib', 'Bib Number': 'Bib', 'Gen': 'Gender',
    'RF In': 'Robinson Flat', 'Robinson': 'Robinson Flat', 'Robinson Flat Arrived': 'Robinson Flat',
    'MB In': 'Michigan Bluff', 'Michigan': 'Michigan Bluff', 'Michigan Bluff Arrive': 'Michigan Bluff',
    'FHill In': 'Foresthill', 'Fhill In': 'Foresthill', 'Foresthill School': 'Foresthill', 'Foresthill School Arrive': 'Foresthill',
    'Elapsed Time': 'Elapsed',
    'Auburn Finish Line': 'Finish_Time',
    'Placer High Finish': 'Finish_Time'
}

In [4]:
# ==========================================
# 2. Time Harmonization Function
# ==========================================
def convert_to_elapsed_minutes(time_str):
    if pd.isna(time_str): return np.nan
    cleaned_str = str(time_str).strip().upper()
    if cleaned_str in ['--:--', '--:--:--', 'DNF', '---', '--', '']: return np.nan
    
    parts = cleaned_str.split(':')
    try:
        if len(parts) == 3: 
            return int(parts[0]) * 60 + int(parts[1]) + float(parts[2]) / 60.0
        elif len(parts) == 2: 
            total_mins = int(parts[0]) * 60 + int(parts[1])
            elapsed = total_mins - 300 
            if elapsed < 0: elapsed += 1440 
            return elapsed
    except ValueError:
        return np.nan
    return np.nan

In [5]:
# ==========================================
# 3. The Resilient Processing Loop
# ==========================================
all_years_data = []
file_paths = glob.glob("wser_raw_data/*.*")

print("Starting Resilient ETL Pipeline...\n")

for file in file_paths:
    year = int(''.join(filter(str.isdigit, file.split('/')[-1])))
    print(f"Processing {year}...")
    
    df_raw = None
    
    # Safely Read the File
    try: df_raw = pd.read_excel(file, header=None)
    except:
        try: df_raw = pd.read_html(file)[0]
        except:
            try: df_raw = pd.read_csv(file, header=None, encoding='latin1')
            except: continue

    if df_raw is None or df_raw.empty or 'ÐÏ' in str(df_raw.iloc[0,0]):
        print(f"  -> ERROR: Cannot read {year} (likely missing xlrd library).")
        continue

    # Dynamic Header Scanner
    header_idx = 0
    for idx, row in df_raw.head(15).iterrows():
        row_values = [str(val).lower().strip() for val in row.values]
        if 'bib' in row_values or 'fname' in row_values or 'bib number' in row_values:
            header_idx = idx
            break
            
    df = df_raw.iloc[header_idx + 1:].copy()
    
    # Clean column headers and drop duplicates (Fixes 2009 Bug)
    df.columns = [str(c).strip() for c in df_raw.iloc[header_idx].values]
    df = df.loc[:, ~df.columns.duplicated(keep='first')]
    
    # Rename using our map and explicitly add the Year (Fixes 2017/2021 Bug)
    df.rename(columns=column_map, inplace=True)
    df['Year'] = year
    
    # Bulletproof Target Variable Extraction (Fixes 2015 Bug)
    finish_col = None
    for col in ['Elapsed', 'Finish', 'Time', 'Finish_Time']:
        if col in df.columns:
            finish_col = col
            break
            
    if finish_col:
        df['Status'] = np.where((df[finish_col] == '--:--') | (df[finish_col].isna()) | (df[finish_col] == 'DNF'), 0, 1)
    else:
        print(f"  -> WARNING: No Finish/Time column found for {year}.")
        continue

    # Extract Age and Gender
    if 'Gender' in df.columns and 'Age' in df.columns:
        df['Runner_Gender'] = df['Gender']
        df['Runner_Age'] = pd.to_numeric(df['Age'], errors='coerce')
    elif 'Div' in df.columns:
        df['Runner_Gender'] = df['Div'].astype(str).str.split('/').str[0]
        df['Runner_Age'] = pd.to_numeric(df['Div'].astype(str).str.split('/').str[1], errors='coerce')
    else:
        df['Runner_Gender'] = np.nan
        df['Runner_Age'] = np.nan

    # Convert checkpoints to minutes
    for cp in ['Robinson Flat', 'Michigan Bluff', 'Foresthill']:
        if cp in df.columns:
            df[f'{cp}_Min'] = df[cp].apply(convert_to_elapsed_minutes)
    
    # Survivorship Filter & Biomechanics
    # First, verify ALL required checkpoints actually exist in this year's data
    core_checkpoints = ['Foresthill_Min', 'Michigan Bluff_Min', 'Robinson Flat_Min']
    
    if all(cp in df.columns for cp in core_checkpoints):
        df_clean = df.dropna(subset=['Foresthill_Min']).copy()
        
        # Now it is safe to do the math!
        df_clean['Pace_Start_to_Robinson'] = df_clean['Robinson Flat_Min'] / 30.3
        df_clean['Pace_Canyons'] = (df_clean['Michigan Bluff_Min'] - df_clean['Robinson Flat_Min']) / 25.4
        df_clean['Pace_Degradation_Canyons'] = (df_clean['Pace_Canyons'] - df_clean['Pace_Start_to_Robinson']) / df_clean['Pace_Start_to_Robinson']
        # Segment 3: Michigan Bluff to Foresthill (Mile 55.7 to 62.0)
        df_clean['Pace_MB_to_FH'] = (df_clean['Foresthill_Min'] - df_clean['Michigan Bluff_Min']) / 6.3
        df_clean['Pace_Degradation_FH'] = (df_clean['Pace_MB_to_FH'] - df_clean['Pace_Canyons']) / df_clean['Pace_Canyons']
        
        features = ['Year', 'Bib', 'Runner_Age', 'Runner_Gender', 
                    'Pace_Start_to_Robinson', 'Pace_Canyons', 'Pace_Degradation_Canyons','Pace_MB_to_FH', 'Pace_Degradation_FH', 'Status']
        
        missing = [col for col in features if col not in df_clean.columns]
        if not missing:
            export_cols = features.copy()
            if 'First Name' in df_clean.columns: export_cols.append('First Name')
            if 'Last Name' in df_clean.columns: export_cols.append('Last Name')
            all_years_data.append(df_clean[export_cols])
        else:
            print(f"  -> WARNING: {year} missing features: {missing}")
    else:
         print(f"  -> WARNING: {year} is missing a core course checkpoint due to a reroute. Skipping.")

Starting Resilient ETL Pipeline...

Processing 2017...
Processing 2021...
Processing 2009...
Processing 2016...
Processing 2025...
Processing 2024...
Processing 2007...
Processing 2013...
Processing 2015...
Processing 2012...
  -> WARNING: 2012 is missing a core course checkpoint due to a reroute. Skipping.
Processing 2006...
Processing 2010...
Processing 2004...
Processing 2019...
Processing 2005...
Processing 2011...
  -> WARNING: 2011 is missing a core course checkpoint due to a reroute. Skipping.
Processing 2023...
Processing 2022...
Processing 2018...
Processing 2014...


In [6]:
# ==========================================
# 4. Final Environmental Merge
# ==========================================
master_runner_df = pd.concat(all_years_data, ignore_index=True)

weather_data = {
    'Year': [2025, 2024, 2023, 2022, 2021, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2007, 2006, 2005, 2004],
    'High_F': [95, 94, 80, 97, 101, 83, 98, 95, 93, 91, 89, 102, 71, 82, 91, 99, 87, 101, 78, 87]
}
snow_data = {
    'Year': [2025, 2024, 2023, 2022, 2021, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2007, 2006, 2005, 2004],
    'June_10_Snow': [0.0, 0.0, 6.3, 0.0, 0.0, 19.6, 0.0, 22.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.9, 17.3, 0.0, 0.0, 18.0, 25.0, 0.0]
}

final_df = pd.merge(master_runner_df, pd.DataFrame(weather_data), on='Year', how='left')
final_df = pd.merge(final_df, pd.DataFrame(snow_data), on='Year', how='left')

final_ml_ready_df = final_df.dropna()
final_ml_ready_df.to_csv("WSER_Master_ML_Dataset.csv", index=False)
print(f"\nSUCCESS! Processed {len(final_ml_ready_df)} total runners into ML dataset.")


SUCCESS! Processed 2959 total runners into ML dataset.
